# Improvement B: receptive field for long series

The LITE paper attributes its weaker results on long series to a small receptive field (RF = 114). Two variants with a larger receptive field are compared with the original LITE on 20 UCR datasets.

| Variant | DWSC layers (kernel, dilation) | RF (formula) | RF (measured) | Parameters (2 classes) |
|---|---|---|---|---|
| BASE (original LITE) | (20, 2), (10, 4) | 114 | 170 | 10,672 |
| D16 | (20, 4), (10, 16) | 260 | 300 | 10,672 |
| L4 | (20, 2), (10, 4), (5, 8) | 146 | 202 | 11,984 |

Only the DWSC layers after the first layer change; the rest is the authors' code. The measured RF includes the fixed hand-crafted filters of the first layer (up to 96 steps), which the formula leaves out.

**Protocol:** 5 runs per variant and dataset, 1500 epochs, same settings as the original. BASE re-scores the reproduction checkpoints when they are in `/kaggle/working/lite_benchmark/LITE/`, otherwise it is retrained.

**Datasets:** 13 long (431–1751 steps), 4 medium (270–350) and 3 short controls (96–251); all equal-length without missing values.

**Setup:** Kaggle, 2 × T4 GPU, internet enabled; same environment as the reproduction notebook.

**Output** (`/kaggle/working/improve_b/`): `results_base.csv`, `results_d16.csv`, `results_l4.csv`, `improve_b_per_dataset.csv`, `improve_b_summary.csv`, `improve_b_per_dataset_significance.csv`.

In [ ]:
# Paths and settings
import os, glob, json, time, subprocess, threading
from pathlib import Path
import numpy as np, pandas as pd

WORK = Path("/kaggle/working")
REPO = WORK / "LITE"
OUT = WORK / "improve_b"
LOGS = OUT / "logs"
LOGS.mkdir(parents=True, exist_ok=True)
REPRO_DIR = WORK / "lite_benchmark" / "LITE"

DATASETS = ["ECG200", "Beef", "BeetleFly", "BirdChicken", "Car", "ArrowHead", "FaceFour",
            "DiatomSizeReduction", "Ham", "Fish", "Adiac", "ACSF1", "CricketX", "Earthquakes",
            "Computers", "FiftyWords", "EOGVerticalSignal", "EOGHorizontalSignal",
            "CinCECGTorso", "EthanolLevel"]
VARIANTS = ["BASE", "D16", "L4"]
RUNS = 5

subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"])

In [ ]:
# Clone LITE (pinned commit) and set n_epochs to 1500
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/MSD-IRIMAS/LITE.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "e62f643"], check=True)

# n_epochs defaults to 2 in this commit
lite_py = REPO / "src/classifiers/lite.py"
code_txt = lite_py.read_text()
if "n_epochs=1500," not in code_txt:
    assert code_txt.count("n_epochs=2,") == 1, "lite.py is not the expected version"
    lite_py.write_text(code_txt.replace("n_epochs=2,", "n_epochs=1500,"))
print([l.strip() for l in lite_py.read_text().splitlines() if "n_epochs=" in l])

In [ ]:
%%writefile /kaggle/working/LITE/improve_b.py
"""
Improvement B: LITE with configurable DWSC dilations.

  BASE: (2, 4)     original LITE (re-scores checkpoints if --baseline-directory is given)
  D16:  (4, 16)    larger dilations, same parameters
  L4:   (2, 4, 8)  one extra DWSC layer

Each finished run writes <out>/<variant>/run_<r>/<dataset>/result.json and ypred.npy;
runs that already have result.json are skipped.
"""
import argparse
import json
import os
import sys
import time

import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score

sys.path.insert(0, os.getcwd())
from src.classifiers import LITE  # noqa: E402
from src.utils import load_data  # noqa: E402

VARIANTS = {
    "BASE": (2, 4),
    "D16": (4, 16),
    "L4": (2, 4, 8),
}

HYBRID_KERNELS = [2, 4, 8, 16, 32, 64]

# measured RF, including the fixed hand-crafted filters of the first layer
RF_MEASURED = {"BASE": 170, "D16": 300, "L4": 202}


def receptive_field(dwsc_dilations, first_kernel=40, include_hybrid=False):
    """Receptive field of the last DWSC layer (DWSC kernels 20, 10, 5, ...)."""
    k0 = first_kernel
    if include_hybrid:
        k0 = max(k0, HYBRID_KERNELS[-1] + HYBRID_KERNELS[-1] // 2)
    rf = k0
    k = first_kernel // 2
    for i, d in enumerate(dwsc_dilations):
        rf += (k // (2 ** i) - 1) * d
    return rf


class LITEVariant(LITE):
    """LITE with one dilation per DWSC layer."""

    def __init__(self, dwsc_dilations, **kwargs):
        self.dwsc_dilations = tuple(dwsc_dilations)
        super().__init__(**kwargs)

    def build_model(self):
        # same as LITE.build_model except for the DWSC loop
        self.keep_track = 0
        input_layer = tf.keras.layers.Input((self.length_TS, self.n_channels))

        inception = self._inception_module(
            input_tensor=input_layer,
            dilation_rate=1,
            use_hybird_layer=self.use_custom_filters,
        )
        self.kernel_size //= 2

        x = inception
        for i, d in enumerate(self.dwsc_dilations):
            x = self._fcn_module(
                input_tensor=x,
                kernel_size=self.kernel_size // (2 ** i),
                n_filters=self.n_filters,
                dilation_rate=d if self.use_dilation else 1,
            )

        gap = tf.keras.layers.GlobalAveragePooling1D()(x)
        output_layer = tf.keras.layers.Dense(units=self.n_classes, activation="softmax")(gap)
        self.model = tf.keras.models.Model(inputs=input_layer, outputs=output_layer)

        reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor="loss", factor=0.5, patience=50, min_lr=1e-4
        )
        model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
            filepath=self.output_directory + "best_model.keras",
            monitor="loss",
            save_best_only=True,
        )
        self.callbacks = [reduce_lr, model_checkpoint]
        self.model.compile(loss="categorical_crossentropy", optimizer="Adam", metrics=["accuracy"])


def get_args():
    p = argparse.ArgumentParser()
    p.add_argument("--dataset", required=True)
    p.add_argument("--variant", required=True, choices=list(VARIANTS))
    p.add_argument("--runs", type=int, default=5)
    p.add_argument("--epochs", type=int, default=1500)
    p.add_argument("--output-directory", required=True)
    p.add_argument("--plot-test", action="store_true",
                   help="evaluate the test set every epoch for the plots, as in main.py")
    p.add_argument("--baseline-directory", default=None,
                   help="folder with run_<r>/<dataset>/best_model.keras to re-score (BASE only)")
    return p.parse_args()


def main():
    a = get_args()
    xtrain, ytrain, xtest, ytest = load_data(file_name=a.dataset)
    length_TS, n_channels = int(xtrain.shape[1]), int(xtrain.shape[2])
    n_classes = len(np.unique(ytrain))
    dil = VARIANTS[a.variant]

    for r in range(a.runs):
        out = os.path.join(a.output_directory, a.variant, f"run_{r}", a.dataset) + "/"
        if os.path.exists(out + "result.json"):
            print(f"{a.variant} run_{r} {a.dataset}: already done, skipping", flush=True)
            continue
        os.makedirs(out, exist_ok=True)

        rescored = a.variant == "BASE" and a.baseline_directory
        if rescored:
            src = os.path.join(a.baseline_directory, f"run_{r}", a.dataset, "best_model.keras")
            if not os.path.exists(src):
                print(f"BASE run_{r} {a.dataset}: no checkpoint at {src}, skipping", flush=True)
                continue
            model = tf.keras.models.load_model(src, compile=False)
            train_time = None
        else:
            clf = LITEVariant(
                dwsc_dilations=dil,
                output_directory=out,
                length_TS=length_TS,
                n_channels=n_channels,
                n_classes=n_classes,
                n_epochs=a.epochs,
            )
            t0 = time.time()
            clf.fit(xtrain=xtrain, ytrain=ytrain, xval=xtest, yval=ytest, plot_test=a.plot_test)
            train_time = time.time() - t0
            model = tf.keras.models.load_model(out + "best_model.keras", compile=False)

        ypred = model.predict(xtest, verbose=0)
        acc = float(accuracy_score(ytest, np.argmax(ypred, axis=1)))
        np.save(out + "ypred.npy", ypred)
        np.save(os.path.join(a.output_directory, a.variant, "ytest_" + a.dataset + ".npy"), ytest)
        res = dict(
            dataset=a.dataset, variant=a.variant, run=r, accuracy=acc,
            params=int(model.count_params()),
            trainable_params=int(sum(np.prod(w.shape) for w in model.trainable_weights)),
            rf=receptive_field(dil), rf_measured=RF_MEASURED[a.variant],
            length=length_TS, n_train=int(len(ytrain)), n_test=int(len(ytest)), n_classes=n_classes,
            epochs=None if rescored else a.epochs, train_time_s=train_time,
        )
        with open(out + "result.json", "w") as f:
            json.dump(res, f)
        print(f"{a.variant} run_{r} {a.dataset}: acc={acc:.4f} params={res['params']} rf={res['rf']}", flush=True)
        tf.keras.backend.clear_session()


if __name__ == "__main__":
    main()

In [ ]:
# Python 3.11 environment
!pip install -q uv
!uv python install 3.11
!uv venv /kaggle/working/lite_env311 --python 3.11 --allow-existing
!UV_LINK_MODE=copy uv pip install -q --python /kaggle/working/lite_env311/bin/python \
    "tensorflow[and-cuda]==2.16.1" "keras==3.6.0" "numpy==1.26.4" "scikit-learn==1.4.2" \
    "pandas==2.0.3" "matplotlib==3.9.0" "codecarbon" "aeon==1.3.0"

In [ ]:
# Environment variables and GPU check
PY = "/kaggle/working/lite_env311/bin/python"
SITE = "/kaggle/working/lite_env311/lib/python3.11/site-packages"
ENV = os.environ.copy()
ENV["LD_LIBRARY_PATH"] = ":".join(sorted(glob.glob(f"{SITE}/nvidia/*/lib")) + [ENV.get("LD_LIBRARY_PATH", "")])
ENV["PATH"] = f"{SITE}/nvidia/cuda_nvcc/bin:" + ENV["PATH"]
ENV["MPLBACKEND"] = "Agg"
ENV["TF_CPP_MIN_LOG_LEVEL"] = "2"
ENV.pop("PYTHONPATH", None)

check = ("import tensorflow as tf, keras, numpy, aeon; g = tf.config.list_physical_devices('GPU'); "
         "print('TF', tf.__version__, '| Keras', keras.__version__, '| NumPy', numpy.__version__, "
         "'| aeon', aeon.__version__, '| GPUs:', len(g))")
r = subprocess.run([PY, "-c", check], env=ENV, capture_output=True, text=True)
print(r.stdout, r.stderr[-1500:])
N_GPUS = int(r.stdout.split("GPUs:")[-1]) if "GPUs:" in r.stdout else 0
assert N_GPUS >= 1, "No GPU found: Settings -> Accelerator -> GPU T4 x2"
GPUS = list(range(N_GPUS))
print('Using GPUs:', GPUS)

In [ ]:
# Download the datasets; check for reproduction checkpoints
prefetch = r"""
import sys
from src.utils import load_data
for d in sys.argv[1:]:
    xtr, ytr, xte, yte = load_data(d)
    print(f"{d:22} length {xtr.shape[1]:5}  train {len(ytr):4}  test {len(yte):5}", flush=True)
"""
r = subprocess.run([PY, "-c", prefetch, *DATASETS], cwd=REPO, env=dict(ENV, CUDA_VISIBLE_DEVICES=""),
                   capture_output=True, text=True)
print(r.stdout)
assert r.returncode == 0, r.stderr[-2000:]

BASE_RESCORE = all((REPRO_DIR / f"run_{k}" / d / "best_model.keras").exists() for d in DATASETS for k in range(RUNS))
print("BASE:", "re-scoring reproduction checkpoints" if BASE_RESCORE else "training the original LITE here")

In [ ]:
# Train on all GPUs (shared task list)
TASKS = [(v, d) for v in VARIANTS for d in DATASETS]
CLAIMED = set()
CLAIM_LOCK, CSV_LOCK = threading.Lock(), threading.Lock()

def done(v, d):
    return all((OUT / v / f"run_{k}" / d / "result.json").exists() for k in range(RUNS))

def update_csv(v, d):
    if not done(v, d):
        return
    scores = [json.loads((OUT / v / f"run_{k}" / d / "result.json").read_text())["accuracy"] for k in range(RUNS)]
    ytest = np.load(OUT / v / f"ytest_{d}.npy")
    ypred = np.mean([np.load(OUT / v / f"run_{k}" / d / "ypred.npy") for k in range(RUNS)], axis=0)
    row = {"dataset": d, "LITE-mean": np.mean(scores), "LITE-std": np.std(scores),
           "LITETime": float((ypred.argmax(axis=1) == ytest).mean())}
    with CSV_LOCK:
        path = OUT / f"results_{v.lower()}.csv"
        old = pd.read_csv(path) if path.exists() else None
        rows = ([] if old is None else old[old.dataset != d].to_dict("records")) + [row]
        df = pd.DataFrame(rows)
        df["order"] = df.dataset.map({x: i for i, x in enumerate(DATASETS)})
        df.sort_values("order").drop(columns="order").to_csv(path, index=False)

def worker(gpu):
    env = dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu), TF_FORCE_GPU_ALLOW_GROWTH="true")
    while True:
        with CLAIM_LOCK:
            task = next((t for t in TASKS if t not in CLAIMED and not done(*t)), None)
            if task is None:
                return
            CLAIMED.add(task)
        v, d = task
        t0 = time.time()
        cmd = [PY, "improve_b.py", "--dataset", d, "--variant", v, "--runs", str(RUNS),
               "--output-directory", str(OUT) + "/"]
        if v == "BASE" and BASE_RESCORE:
            cmd += ["--baseline-directory", str(REPRO_DIR) + "/"]
        else:
            cmd += ["--plot-test"]
        with open(LOGS / f"{v}_{d}.log", "a") as f:
            rc = subprocess.run(cmd, cwd=REPO, env=env, stdout=f, stderr=subprocess.STDOUT).returncode
        if rc == 0 and done(v, d):
            update_csv(v, d)
            acc = np.mean([json.loads(p.read_text())["accuracy"] for p in OUT.glob(f"{v}/run_*/{d}/result.json")])
            print(time.strftime("%d %b %H:%M") + f"  GPU{gpu}  {v:4} {d:22} {(time.time()-t0)/60:6.1f} min  mean acc {acc:.4f}", flush=True)
        else:
            print(f"GPU{gpu}  {v} {d} FAILED - see {LOGS / f'{v}_{d}.log'}", flush=True)

for v in VARIANTS:
    for d in DATASETS:
        update_csv(v, d)

print(f"{sum(not done(*t) for t in TASKS)} of {len(TASKS)} tasks left, GPUs {GPUS}", flush=True)
threads = [threading.Thread(target=worker, args=(g,), daemon=True) for g in GPUS]
for t in threads:
    t.start()
for t in threads:
    t.join()
print("All Improvement B tasks finished.")

In [ ]:
# Results
from scipy.stats import wilcoxon, spearmanr

Rr = pd.DataFrame([json.loads(p.read_text()) for p in OUT.glob("*/run_*/*/result.json")])
T = Rr.pivot_table(index=["dataset", "length", "n_train"], columns="variant", values="accuracy", aggfunc="mean").reset_index()
T = T.sort_values("length")
NEW = [v for v in VARIANTS if v != "BASE"]
for v in NEW:
    if v in T:
        T[f"gain_{v}"] = T[v] - T["BASE"]
pd.set_option("display.float_format", "{:.4f}".format)
print("Accuracy per dataset (mean of 5 runs; BASE = original LITE)")
display(T)

rows = []
for v in NEW:
    if v not in T:
        continue
    t = T.dropna(subset=[v, "BASE"])
    g = t[f"gain_{v}"]
    rows.append({"variant": v, "datasets": len(t), "BASE mean": t.BASE.mean(), "variant mean": t[v].mean(),
                 "better/same/worse": f"{(g > 1e-9).sum()}/{(g.abs() <= 1e-9).sum()}/{(g < -1e-9).sum()}",
                 "Wilcoxon p": wilcoxon(t[v], t.BASE).pvalue if (g.abs() > 1e-9).sum() >= 5 else np.nan,
                 "Spearman(length, gain)": spearmanr(t.length, g)[0] if len(t) >= 5 else np.nan,
                 "train min/run": Rr[Rr.variant == v].train_time_s.mean() / 60})
print("\nSummary vs original LITE")
display(pd.DataFrame(rows))

T["group"] = pd.cut(T.length, [0, 300, 1000, 10**6], right=False,
                    labels=["short (<300)", "medium (300-1000)", "long (>=1000)"])
print("\nMean accuracy by series length")
display(T.groupby("group", observed=True)[[c for c in VARIANTS if c in T]].mean())
print("\nArchitecture (2-class dataset)")
display(Rr[Rr.n_classes == 2].groupby("variant")[["params", "trainable_params", "rf", "rf_measured"]].first())

T.drop(columns="group").to_csv(OUT / "improve_b_per_dataset.csv", index=False)
pd.DataFrame(rows).to_csv(OUT / "improve_b_summary.csv", index=False)
print("\nSaved to", OUT)

In [ ]:
# Per-dataset significance (Welch t-test, 5 vs 5 runs)
from scipy.stats import ttest_ind

rows = []
for d, g in Rr.groupby("dataset"):
    base = g[g.variant == "BASE"].accuracy.values
    for v in NEW:
        x = g[g.variant == v].accuracy.values
        if len(x) < RUNS or len(base) < RUNS:
            continue
        diff = x.mean() - base.mean()
        p = (1.0 if diff == 0 else 0.0) if np.std(x) == 0 and np.std(base) == 0 else ttest_ind(x, base, equal_var=False).pvalue
        verdict = ("significantly BETTER" if p < 0.05 and diff > 0 else
                   "significantly WORSE" if p < 0.05 and diff < 0 else "no clear difference")
        rows.append(dict(dataset=d, length=g.length.iloc[0], variant=v, BASE=base.mean(), new=x.mean(),
                         gain_points=100 * diff, p_value=p, verdict=verdict))
S = pd.DataFrame(rows).sort_values(["variant", "gain_points"], ascending=[True, False])
for v in NEW:
    t = S[S.variant == v]
    print(f"\n{v}: significantly better {t.verdict.str.contains('BETTER').sum()}, "
          f"significantly worse {t.verdict.str.contains('WORSE').sum()}, no clear difference {(t.verdict == 'no clear difference').sum()}")
    display(t.drop(columns="variant"))
print(f"\nNote: {len(S)} tests at p < 0.05, so about {0.05 * len(S):.0f} could be significant by chance.")
S.to_csv(OUT / "improve_b_per_dataset_significance.csv", index=False)